# Post-test: sentence.txt end-to-end evaluation

這個 notebook 會：
1. 讀取 `sentence.txt` 中的句子與 gloss 編號。
2. 從 `27261843/splits/metadata.csv` 找到對應的孤立手語片段（預設優先 `test/front`）。
3. 對每個 gloss 片段重新做 MediaPipe landmark 抽取。
4. 使用 `27261843/checkpoints/best_model.pth` 做 Top-5 手語辨識。
5. 把每個位置的 Top-5 候選 + 上下文送到可配置的 LLM API，重組為自然中文句子。
6. 輸出 token-level 與 sentence-level 評估結果。

> 設計參考：`Chinese_Sign_Language_baseFront_End_(1).ipynb` + `CSL_Training_Pipeline(best).ipynb`。

In [ ]:
!pip install -q mediapipe==0.10.14 opencv-python-headless requests pandas matplotlib opencc-python-reimplemented
!pip install -q --force-reinstall "protobuf==4.25.9"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 9.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opent

In [ ]:
import os
import re
import json
import math
from pathlib import Path
from functools import lru_cache

import cv2
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import mediapipe as mp
import matplotlib.pyplot as plt

from opencc import OpenCC
from IPython.display import display, Markdown

from google.colab import drive
drive.mount('/content/drive')


def first_existing(candidates, description):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Cannot find {description}. Tried: {candidates}')


def looks_like_data_root(path):
    path = Path(path)
    return path.is_dir() and (path / 'splits' / 'metadata.csv').exists() and (path / 'checkpoints' / 'best_model.pth').exists()


def find_data_root():
    direct_candidates = [
        '/content/drive/MyDrive/4016project/27261843',
        '/content/drive/MyDrive/27261843',
        './27261843',
        '27261843'
    ]
    for candidate in direct_candidates:
        candidate = Path(candidate)
        if looks_like_data_root(candidate):
            return candidate.resolve()

    search_roots = [
        Path.cwd(),
        Path('/content/drive/MyDrive'),
        Path('/content/drive'),
        Path('/content')
    ]
    visited = set()
    for root in search_roots:
        root = Path(root)
        root_key = root.as_posix()
        if root_key in visited or not root.exists():
            continue
        visited.add(root_key)
        try:
            matches = [p for p in root.rglob('27261843') if looks_like_data_root(p)]
        except Exception:
            matches = []
        if matches:
            matches = sorted(matches, key=lambda p: (0 if 'MyDrive' in p.as_posix() else 1, len(p.parts)))
            return matches[0].resolve()

    extra_msg = ''
    if not Path('/content/drive').exists():
        extra_msg = ' It looks like Google Drive is not mounted yet; please mount Drive first.'
    raise FileNotFoundError(
        'Cannot find DATA_ROOT. Expected a folder named 27261843 containing splits/metadata.csv and checkpoints/best_model.pth.' + extra_msg
    )


def find_sentence_file(project_root, data_root):
    direct_candidates = [
        project_root / 'sentence.txt',
        data_root / 'sentence.txt',
        Path.cwd() / 'sentence.txt',
        Path('sentence.txt')
    ]
    for candidate in direct_candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate.resolve()

    search_roots = [project_root, data_root, Path.cwd()]
    visited = set()
    for root in search_roots:
        root = Path(root)
        root_key = root.as_posix()
        if root_key in visited or not root.exists():
            continue
        visited.add(root_key)
        try:
            matches = sorted(root.rglob('sentence.txt'))
        except Exception:
            matches = []
        if matches:
            return matches[0].resolve()

    raise FileNotFoundError('Cannot find sentence.txt near the project root, DATA_ROOT, or current working directory.')


DATA_ROOT = find_data_root()
PROJECT_ROOT = DATA_ROOT.parent
SPLITS_DIR = DATA_ROOT / 'splits'
CHECKPOINT_PATH = DATA_ROOT / 'checkpoints' / 'best_model.pth'
SENTENCE_FILE = find_sentence_file(PROJECT_ROOT, DATA_ROOT)

LLM_API_URL = os.getenv('LLM_API_URL', 'https://api.deepseek.com/v1/chat/completions')
LLM_MODEL = os.getenv('LLM_MODEL', 'deepseek-chat')
LLM_API_KEY = os.getenv('LLM_API_KEY', 'sk-e8cf508675ce4c0f8b27d8b8665c402a')
LLM_TEMPERATURE = float(os.getenv('LLM_TEMPERATURE', '0.2'))
USE_LLM_JUDGE = False

PREFERRED_SPLIT = 'test'
PREFERRED_VIEW = 'front'
PREFERRED_SIGNER = '10'
DEFAULT_MAX_LEN = 64

cc_t2s = OpenCC('t2s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CWD            :', Path.cwd())
print('DATA_ROOT      :', DATA_ROOT)
print('SENTENCE_FILE  :', SENTENCE_FILE)
print('CHECKPOINT     :', CHECKPOINT_PATH)
print('Device         :', device)
print('LLM API URL    :', LLM_API_URL)
print('LLM model      :', LLM_MODEL)
print('LLM key set    :', bool(LLM_API_KEY))

Mounted at /content/drive
CWD            : /content
DATA_ROOT      : /content/drive/MyDrive/4016project/27261843
SENTENCE_FILE  : /content/drive/MyDrive/4016project/sentence.txt
CHECKPOINT     : /content/drive/MyDrive/4016project/27261843/checkpoints/best_model.pth
Device         : cpu
LLM API URL    : https://api.deepseek.com/v1/chat/completions
LLM model      : deepseek-chat
LLM key set    : True


In [ ]:
metadata_df = pd.read_csv(SPLITS_DIR / 'metadata.csv', dtype={'gloss_id': str, 'signer_id': str})
metadata_df['gloss_id'] = metadata_df['gloss_id'].str.zfill(4)
metadata_df['signer_id'] = metadata_df['signer_id'].astype(str).str.zfill(2)
metadata_df['view'] = metadata_df['view'].astype(str)
metadata_df['split'] = metadata_df['split'].astype(str)

id_to_zh = {str(row.gloss_id).zfill(4): str(row.gloss_zh) for row in metadata_df.itertuples()}
id_to_en = {str(row.gloss_id).zfill(4): str(row.gloss_en) for row in metadata_df.itertuples()}


def parse_sentence_file(path):
    items = []
    for line_no, raw_line in enumerate(Path(path).read_text(encoding='utf-8').splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue
        ids = [m.zfill(4) for m in re.findall(r'\b\d{4}\b', line)]
        if not ids:
            continue
        first_match = re.search(r'\b\d{4}\b', line)
        reference_text = line[:first_match.start()].strip() if first_match else ''
        reference_text = reference_text.rstrip(' ->')
        items.append({
            'sentence_id': line_no,
            'reference_text': reference_text,
            'gloss_ids': ids,
            'raw_line': raw_line
        })
    return items


sentence_items = parse_sentence_file(SENTENCE_FILE)
preview_rows = []
for item in sentence_items:
    preview_rows.append({
        'sentence_id': item['sentence_id'],
        'reference_text': item['reference_text'],
        'n_glosses': len(item['gloss_ids']),
        'gloss_ids': ' -> '.join(item['gloss_ids'])
    })

display(pd.DataFrame(preview_rows))
print(f'Total sentence items: {len(sentence_items)}')

,sentence_id,reference_text,n_glosses,gloss_ids
0,1,我喜歡蘋果,3,1928 -> 2462 -> 0514
1,2,母親買麵包,3,2832 -> 3993 -> 2985
2,3,老師幫助學生,3,1597 -> 5094 -> 0526
3,4,朋友找醫生,3,5008 -> 1256 -> 2102
4,5,明天大家休息,3,4542 -> 5599 -> 6105
5,6,昨天考試緊張,3,4466 -> 1962 -> 6363
6,7,今天我去超市買牛奶和雞蛋,8,1770 -> 1928 -> 3703 -> 1823 -> 3993 -> 0968 -...
7,8,因為大雨，父親和伯父坐在家喝茶,10,5780 -> 0639 -> 0396 -> 5478 -> 1818 -> 2732 -...


Total sentence items: 8


In [ ]:
POSE_DIM = 33 * 3
LHAND_DIM = 21 * 3
RHAND_DIM = 21 * 3
RAW_FEATURE_DIM = POSE_DIM + LHAND_DIM + RHAND_DIM
POSE_FEATURE_DIM = POSE_DIM * 2
LHAND_FEATURE_DIM = LHAND_DIM * 2
RHAND_FEATURE_DIM = RHAND_DIM * 2
FEATURE_DIM = POSE_FEATURE_DIM + LHAND_FEATURE_DIM + RHAND_FEATURE_DIM

LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_WRIST = 15
RIGHT_WRIST = 16


def normalize_relative_landmarks(seq):
    seq = np.asarray(seq, dtype=np.float32)
    T = seq.shape[0]
    pose = seq[:, :POSE_DIM].reshape(T, 33, 3)
    lhand = seq[:, POSE_DIM:POSE_DIM + LHAND_DIM].reshape(T, 21, 3)
    rhand = seq[:, POSE_DIM + LHAND_DIM:].reshape(T, 21, 3)

    shoulder_center = 0.5 * (pose[:, LEFT_SHOULDER] + pose[:, RIGHT_SHOULDER])
    shoulder_scale = np.linalg.norm(pose[:, LEFT_SHOULDER] - pose[:, RIGHT_SHOULDER], axis=1, keepdims=True)
    shoulder_scale = np.clip(shoulder_scale, 1e-3, None)

    pose_rel = (pose - shoulder_center[:, None, :]) / shoulder_scale[:, None, :]
    lhand_rel = (lhand - pose[:, LEFT_WRIST:LEFT_WRIST + 1, :]) / shoulder_scale[:, None, :]
    rhand_rel = (rhand - pose[:, RIGHT_WRIST:RIGHT_WRIST + 1, :]) / shoulder_scale[:, None, :]

    feat = np.concatenate([pose_rel.reshape(T, -1), lhand_rel.reshape(T, -1), rhand_rel.reshape(T, -1)], axis=1)
    return np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def append_velocity_features(seq):
    velocity = np.diff(seq, axis=0, prepend=seq[:1])
    feat = np.concatenate([seq, velocity], axis=1)
    return np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class BranchProjector(nn.Module):
    def __init__(self, input_dim, branch_dim, dropout=0.2):
        super().__init__()
        hidden_dim = max(branch_dim * 2, 96)
        self.norm = nn.LayerNorm(input_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, branch_dim)
        )

    def forward(self, x):
        return self.net(self.norm(x))


class SkeletonAwareEmbedding(nn.Module):
    def __init__(self, d_model=288, dropout=0.2):
        super().__init__()
        if d_model % 3 != 0:
            raise ValueError('d_model must be divisible by 3 for pose/left/right branches')
        branch_dim = d_model // 3
        self.pose_proj = BranchProjector(POSE_FEATURE_DIM, branch_dim, dropout)
        self.lhand_proj = BranchProjector(LHAND_FEATURE_DIM, branch_dim, dropout)
        self.rhand_proj = BranchProjector(RHAND_FEATURE_DIM, branch_dim, dropout)
        self.fuse = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        pose = x[:, :, :POSE_FEATURE_DIM]
        lhand = x[:, :, POSE_FEATURE_DIM:POSE_FEATURE_DIM + LHAND_FEATURE_DIM]
        rhand = x[:, :, POSE_FEATURE_DIM + LHAND_FEATURE_DIM:]
        fused = torch.cat([self.pose_proj(pose), self.lhand_proj(lhand), self.rhand_proj(rhand)], dim=-1)
        return fused + self.fuse(fused)


class AttentionPooling(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        hidden_dim = max(d_model // 2, 96)
        self.score = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, pad_mask):
        scores = self.score(x).squeeze(-1)
        scores = scores.masked_fill(pad_mask, -1e4)
        attn = torch.softmax(scores, dim=1)
        return torch.sum(x * attn.unsqueeze(-1), dim=1)


class CSLTransformer(nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, d_model=288, nhead=6, num_layers=4, dim_feedforward=576, dropout=0.3, num_classes=196, max_len=64):
        super().__init__()
        self.embedding = SkeletonAwareEmbedding(d_model=d_model, dropout=dropout * 0.7)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout * 0.5, max_len=max_len + 10)
        self.input_dropout = nn.Dropout(dropout * 0.35)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, enable_nested_tensor=False)
        self.out_norm = nn.LayerNorm(d_model)
        self.attn_pool = AttentionPooling(d_model, dropout=dropout * 0.5)
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def _make_padding_mask(self, lengths, max_len):
        return torch.arange(max_len, device=lengths.device).unsqueeze(0) >= lengths.unsqueeze(1)

    def forward(self, x, lengths):
        x = self.embedding(x)
        x = self.pos_enc(x)
        x = self.input_dropout(x)
        T = x.size(1)
        pad_mask = self._make_padding_mask(lengths, T)
        x = self.transformer(x, src_key_padding_mask=pad_mask)
        mask_f = (~pad_mask).float().unsqueeze(-1)
        mean_x = (x * mask_f).sum(dim=1) / lengths.float().unsqueeze(1).clamp(min=1.0)
        attn_x = self.attn_pool(x, pad_mask)
        mean_x = self.out_norm(mean_x)
        attn_x = self.out_norm(attn_x)
        pooled = torch.cat([mean_x, attn_x], dim=1)
        logits = self.classifier(pooled)
        return logits


ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
config = ckpt.get('config', {})
label_map = {str(k).zfill(4): int(v) for k, v in ckpt['label_map'].items()}
idx2gloss = {int(v): str(k).zfill(4) for k, v in label_map.items()}
NUM_CLASSES = int(ckpt.get('num_classes', len(label_map)))
MAX_LEN = int(config.get('max_len', DEFAULT_MAX_LEN))
D_MODEL = int(config.get('d_model', 384))
NHEAD = int(config.get('nhead', 6))
NUM_LAYERS = int(config.get('num_layers', 4))
DIM_FEEDFORWARD = int(config.get('dim_feedforward', D_MODEL * 2))
DROPOUT = float(config.get('dropout', 0.35))

model = CSLTransformer(
    input_dim=FEATURE_DIM,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES,
    max_len=MAX_LEN
).to(device)

state_key = 'ema_state' if 'ema_state' in ckpt else 'model_state'
model.load_state_dict(ckpt[state_key], strict=True)
model.eval()

print('Checkpoint state key :', state_key)
print('Num classes          :', NUM_CLASSES)
print('Model config         :', {
    'd_model': D_MODEL,
    'nhead': NHEAD,
    'num_layers': NUM_LAYERS,
    'dim_feedforward': DIM_FEEDFORWARD,
    'dropout': DROPOUT,
    'max_len': MAX_LEN
})

Checkpoint state key : ema_state
Num classes          : 200
Model config         : {'d_model': 384, 'nhead': 6, 'num_layers': 4, 'dim_feedforward': 768, 'dropout': 0.35, 'max_len': 64}


In [ ]:
mp_holistic = mp.solutions.holistic


def resolve_case_insensitive_path(base_path, relative_path):
    base_path = Path(base_path)
    current = base_path
    for part in Path(relative_path).parts:
        direct = current / part
        if direct.exists():
            current = direct
            continue
        if not current.exists():
            raise FileNotFoundError(f'Missing parent path: {current}')
        match = None
        part_lower = part.lower()
        for child in current.iterdir():
            if child.name.lower() == part_lower:
                match = child
                break
        if match is None:
            raise FileNotFoundError(f'Cannot resolve path part {part} under {current}')
        current = match
    return current


def clean_gloss_variant_markers(text):
    text = '' if text is None else str(text)
    if not text:
        return ''
    text = re.sub(r'\d+(?:-\d+)+', '', text)
    return text.strip()


def sanitize_generated_text(text):
    text = '' if text is None else str(text)
    if not text:
        return ''

    text = text.replace('```', ' ').replace('`', ' ').strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[-1]

    text = re.sub(r'^(?:最終句子|最终句子|最終答案|最终答案|答案|輸出|输出|句子|重組結果|重组结果|result|final)\s*[:：]\s*', '', text, flags=re.IGNORECASE)
    text = cc_t2s.convert(text)
    text = clean_gloss_variant_markers(text)
    text = re.sub(r"[“”\"']", '', text)
    text = re.sub(r"[，。！？、；：,!.?;:()（）\[\]{}<>《》【】\s]+", '', text)
    return text.strip()


def gloss_to_text(gloss_id):
    gloss_id = str(gloss_id).zfill(4)
    return clean_gloss_variant_markers(id_to_zh.get(gloss_id, gloss_id))


def select_clip_record(gloss_id, preferred_split=PREFERRED_SPLIT, preferred_view=PREFERRED_VIEW, preferred_signer=PREFERRED_SIGNER):
    gloss_id = str(gloss_id).zfill(4)
    subset = metadata_df[metadata_df['gloss_id'] == gloss_id].copy()
    if subset.empty:
        raise ValueError(f'No clip found for gloss_id={gloss_id}')

    def score_row(row):
        score = 0
        if row['view'] == preferred_view:
            score += 100
        if row['split'] == preferred_split:
            score += 50
        if row['signer_id'] == str(preferred_signer).zfill(2):
            score += 20
        score += min(int(row['n_frames']), 50)
        return score

    subset['priority'] = subset.apply(score_row, axis=1)
    subset = subset.sort_values(['priority', 'n_frames'], ascending=[False, False])
    row = subset.iloc[0].to_dict()
    row['gloss_id'] = str(row['gloss_id']).zfill(4)
    row['signer_id'] = str(row['signer_id']).zfill(2)
    return row


def extract_frame_vector(results):
    frame = []

    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark[:33]:
            frame.extend([lm.x, lm.y, lm.z])
    else:
        frame.extend([0.0] * POSE_DIM)

    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark[:21]:
            frame.extend([lm.x, lm.y, lm.z])
    else:
        frame.extend([0.0] * LHAND_DIM)

    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark[:21]:
            frame.extend([lm.x, lm.y, lm.z])
    else:
        frame.extend([0.0] * RHAND_DIM)

    return np.asarray(frame, dtype=np.float32)


@lru_cache(maxsize=256)
def get_sorted_frame_paths(clip_dir_str):
    clip_dir = Path(clip_dir_str)
    frame_paths = sorted(list(clip_dir.glob('*.jpg')) + list(clip_dir.glob('*.jpeg')) + list(clip_dir.glob('*.png')))
    return tuple(str(p) for p in frame_paths)


@lru_cache(maxsize=256)
def extract_raw_landmarks_from_clip(clip_dir_str):
    frame_paths = get_sorted_frame_paths(clip_dir_str)
    if not frame_paths:
        raise FileNotFoundError(f'No frames found in {clip_dir_str}')

    collected = []
    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:
        for frame_path in frame_paths:
            frame = cv2.imread(frame_path)
            if frame is None:
                continue
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image)
            collected.append(extract_frame_vector(results))

    if not collected:
        raise RuntimeError(f'Failed to extract landmarks from {clip_dir_str}')

    seq = np.asarray(collected, dtype=np.float32)
    return np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def prepare_model_input_from_clip(clip_dir):
    raw_seq = extract_raw_landmarks_from_clip(str(clip_dir))
    seq = normalize_relative_landmarks(raw_seq)
    seq = append_velocity_features(seq)
    if seq.shape[0] > MAX_LEN:
        seq = seq[:MAX_LEN]
    length = int(seq.shape[0])
    feat = torch.from_numpy(seq).float().unsqueeze(0).to(device)
    lengths = torch.tensor([length], dtype=torch.long, device=device)
    return feat, lengths, raw_seq.shape[0]


@torch.no_grad()
def predict_topk_from_clip(clip_dir, topk=5):
    feat, lengths, raw_frames = prepare_model_input_from_clip(clip_dir)
    logits = model(feat, lengths)
    probs = torch.softmax(logits, dim=1)[0]
    values, indices = probs.topk(min(topk, probs.numel()))

    items = []
    for prob, idx in zip(values.tolist(), indices.tolist()):
        gloss_id = idx2gloss[int(idx)]
        items.append({
            'rank': len(items) + 1,
            'label_idx': int(idx),
            'gloss_id': gloss_id,
            'gloss_zh': gloss_to_text(gloss_id),
            'prob': float(prob)
        })
    return items, raw_frames, int(lengths.item())


def recognize_sentence_item(sentence_item):
    token_results = []
    for pos, target_gloss_id in enumerate(sentence_item['gloss_ids'], start=1):
        record = select_clip_record(target_gloss_id)
        clip_dir = resolve_case_insensitive_path(DATA_ROOT, record['path'])
        top5, raw_frames, used_frames = predict_topk_from_clip(clip_dir, topk=5)
        token_results.append({
            'position': pos,
            'target_gloss_id': target_gloss_id,
            'target_gloss_zh': gloss_to_text(target_gloss_id),
            'sample_id': record['sample_id'],
            'split': record['split'],
            'view': record['view'],
            'signer_id': record['signer_id'],
            'clip_dir': str(clip_dir),
            'raw_frames': raw_frames,
            'used_frames': used_frames,
            'top5': top5,
            'top1_hit': top5[0]['gloss_id'] == target_gloss_id,
            'top5_hit': target_gloss_id in [x['gloss_id'] for x in top5]
        })
    return token_results

In [ ]:
def format_token_candidates(token_results):
    blocks = []
    for token in token_results:
        header = f"位置 {token['position']}（片段來源: signer {token['signer_id']} / {token['split']} / {token['view']}）"
        cand_lines = []
        for cand in token['top5']:
            cand_lines.append(f"  - Top{cand['rank']}: {cand['gloss_id']} / {cand['gloss_zh']} / p={cand['prob']:.4f}")
        blocks.append(header + '\n' + '\n'.join(cand_lines))
    return '\n\n'.join(blocks)


def generate_candidate_paths(token_results, beam_width=120, top_paths=24):
    beams = [([], 0.0)]
    for token in token_results:
        new_beams = []
        for seq, score in beams:
            for cand in token['top5']:
                prob = max(float(cand['prob']), 1e-8)
                new_beams.append((seq + [cand], score + math.log(prob)))
        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:beam_width]

    paths = []
    seen = set()
    for seq, score in beams:
        sequence_text = ''.join(c['gloss_zh'] for c in seq)
        if sequence_text in seen:
            continue
        seen.add(sequence_text)
        detail = ' / '.join([f"pos{i+1}:Top{c['rank']} {c['gloss_zh']}({c['prob']:.3f})" for i, c in enumerate(seq)])
        paths.append({'sequence': sequence_text, 'score': score, 'detail': detail})
        if len(paths) >= top_paths:
            break
    return paths


def format_candidate_paths(paths):
    if not paths:
        return ''
    lines = []
    for i, path in enumerate(paths, start=1):
        lines.append(f"Path {i}: {path['sequence']} | {path['detail']}")
    return '\n'.join(lines)


def call_openai_compatible_llm(messages, temperature=0.2, max_tokens=300):
    if not LLM_API_KEY:
        return None, 'LLM_API_KEY is empty; no real LLM call was made'

    try:
        response = requests.post(
            LLM_API_URL,
            headers={
                'Authorization': f'Bearer {LLM_API_KEY}',
                'Content-Type': 'application/json'
            },
            json={
                'model': LLM_MODEL,
                'messages': messages,
                'temperature': temperature,
                'max_tokens': max_tokens
            },
            timeout=120
        )
        response.raise_for_status()
        data = response.json()
        content = data['choices'][0]['message']['content'].strip()
        return content, None
    except Exception as exc:
        return None, str(exc)


def reconstruct_sentence_with_llm(sentence_item, token_results):
    top1_sequence = [token['top5'][0]['gloss_zh'] for token in token_results]
    fallback_sentence = ''.join(top1_sequence)
    candidate_paths = generate_candidate_paths(token_results, beam_width=120, top_paths=24)

    system_prompt = (
        '你是專業的中國手語翻譯助手。你現在做的是整句級 Top-5 候選推斷，而不是逐位置直接取 Top1。'
        '你只能使用每個位置給出的 Top-5 候選與百分比作推斷，不能使用任何標準答案，也不能猜測不在 Top-5 裡的新詞。'
        'Top1 很可能是錯的，所以你要聯合比較所有位置的候選，必要時可選 Top2、Top3、Top4、Top5。'
        '你必須先想清楚每個 position 最佳候選是什麼，再決定整句。'
        '如果候選詞中帶有數據集變體標記，例如 1-1、1-2，請把它們視為同一個詞並在最終輸出中刪除。'
        '不要輸出句號、逗號、空格或任何標點；也不要添加超出候選之外的新內容。只輸出一句最終句子，不要解釋。'
    )

    user_prompt = (
        f"這是一句由 {len(token_results)} 個孤立手語片段組成的輸入。\n"
        f"請先在每個 position 的候選裡做最合理選擇，再看整句是否通順。\n"
        f"如果某個位置的 Top1 很怪，但 Top2~Top5 有更符合上下文的詞，就應該改選低排名候選。\n\n"
        f"Top1 baseline（僅供參考，可能錯很多）：{fallback_sentence}\n\n"
        f"逐位置 Top-5 候選：\n{format_token_candidates(token_results)}\n\n"
        f"高分候選路徑（按 joint probability 排序，只作整句比較參考）：\n{format_candidate_paths(candidate_paths)}\n\n"
        f"補充要求：\n"
        f"1. 不要盲目複製 Top1，要優先考慮整句是否合理。\n"
        f"2. 對每個 position 都要選一個最合理候選。\n"
        f"3. 不要輸出句號、逗號、空格或任何標點。\n"
        f"4. 不要添加 Top-5 候選之外的新詞。\n"
        f"5. 如果只有 3 個 position，通常不應輸出超過 3 個內容詞。\n"
        f"6. 允許重排語序，但最終句子必須盡量由各位置候選重組得到。\n"
        f"7. 對於像『老師幫助學生』這種句子，要比較三個 position 的整體語義，而不是各自貪心取最高概率。\n"
        f"8. 如果候選詞帶有像 1-1、1-2 這種 dataset 後綴，請自動忽略，不要原樣輸出。\n"
        f"9. 只輸出一句最終句子。\n"
    )

    content, error = call_openai_compatible_llm([
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt}
    ], temperature=LLM_TEMPERATURE, max_tokens=180)

    if error is not None:
        return {
            'sentence': fallback_sentence,
            'llm_error': error,
            'used_fallback': True
        }

    content = sanitize_generated_text(content.strip().strip(chr(34)).strip())
    return {
        'sentence': content,
        'llm_error': None,
        'used_fallback': False
    }


def normalize_eval_text(text):
    text = '' if text is None else str(text)
    text = cc_t2s.convert(text)
    text = sanitize_generated_text(text)
    return text.strip()


def edit_distance(a, b):
    a = list(a)
    b = list(b)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )
    return dp[-1][-1]


def judge_with_llm(reference, prediction):
    if (not USE_LLM_JUDGE) or (not LLM_API_KEY):
        return None

    prompt = (
        '你是翻譯評分員。請根據語意接近度，對 prediction 與 reference 的一致性打分 1 到 5。'
        '5 表示語意幾乎完全一致，1 表示完全不相關。只輸出一個整數。\n\n'
        f'reference: {reference}\n'
        f'prediction: {prediction}'
    )
    content, error = call_openai_compatible_llm([{'role': 'user', 'content': prompt}], temperature=0.0, max_tokens=10)
    if error is not None or content is None:
        return None
    match = re.search(r'[1-5]', content)
    return int(match.group()) if match else None


def evaluate_sentence(reference_text, predicted_sentence, token_results):
    top1_acc = float(np.mean([token['top1_hit'] for token in token_results]))
    top5_acc = float(np.mean([token['top5_hit'] for token in token_results]))
    reference_norm = normalize_eval_text(reference_text)
    predicted_norm = normalize_eval_text(predicted_sentence)
    dist = edit_distance(reference_norm, predicted_norm)
    char_similarity = 1.0 - dist / max(len(reference_norm), len(predicted_norm), 1)
    contains_reference = int(reference_norm in predicted_norm) if reference_norm else 0
    contains_prediction = int(predicted_norm in reference_norm) if predicted_norm else 0
    llm_score = judge_with_llm(reference_text, predicted_sentence)
    return {
        'reference_text': reference_text,
        'predicted_sentence': predicted_sentence,
        'reference_norm': reference_norm,
        'predicted_norm': predicted_norm,
        'exact_match': int(reference_norm == predicted_norm),
        'contains_reference': contains_reference,
        'contains_prediction': contains_prediction,
        'char_edit_distance': int(dist),
        'char_similarity': float(char_similarity),
        'token_top1_acc': top1_acc,
        'token_top5_acc': top5_acc,
        'llm_semantic_score': llm_score
    }

In [ ]:
all_sentence_rows = []
detailed_results = []

for sentence_item in sentence_items:
    print('=' * 90)
    print(f"Sentence #{sentence_item['sentence_id']}")
    print('Reference :', sentence_item['reference_text'])
    print('Gloss IDs :', ' -> '.join(sentence_item['gloss_ids']))

    token_results = recognize_sentence_item(sentence_item)
    llm_result = reconstruct_sentence_with_llm(sentence_item, token_results)
    eval_result = evaluate_sentence(sentence_item['reference_text'], llm_result['sentence'], token_results)

    top1_glosses = [token['top5'][0]['gloss_zh'] for token in token_results]
    top1_ids = [token['top5'][0]['gloss_id'] for token in token_results]

    print('Top1 gloss       :', ' / '.join([f"{gid}:{gzh}" for gid, gzh in zip(top1_ids, top1_glosses)]))
    print('LLM output       :', llm_result['sentence'])
    print('Reference norm   :', eval_result['reference_norm'])
    print('Prediction norm  :', eval_result['predicted_norm'])
    print('Top1 acc         :', f"{eval_result['token_top1_acc']:.2%}")
    print('Top5 acc         :', f"{eval_result['token_top5_acc']:.2%}")
    print('Char sim         :', f"{eval_result['char_similarity']:.2%}")
    print('Contains ref     :', bool(eval_result['contains_reference']))
    if llm_result['llm_error']:
        print('LLM status : fallback used ->', llm_result['llm_error'])
    else:
        print('LLM status : API success')

    token_table = []
    for token in token_results:
        token_table.append({
            'position': token['position'],
            'target_gloss_id': token['target_gloss_id'],
            'target_gloss_zh': token['target_gloss_zh'],
            'top1_id': token['top5'][0]['gloss_id'],
            'top1_zh': token['top5'][0]['gloss_zh'],
            'top1_prob': round(token['top5'][0]['prob'], 4),
            'top5_ids': ' | '.join([cand['gloss_id'] for cand in token['top5']]),
            'top5_zh': ' | '.join([cand['gloss_zh'] for cand in token['top5']]),
            'top1_hit': token['top1_hit'],
            'top5_hit': token['top5_hit'],
            'sample_id': token['sample_id']
        })
    token_df = pd.DataFrame(token_table)
    display(token_df)

    row = {
        'sentence_id': sentence_item['sentence_id'],
        'reference_text': sentence_item['reference_text'],
        'predicted_sentence': llm_result['sentence'],
        'reference_norm': eval_result['reference_norm'],
        'predicted_norm': eval_result['predicted_norm'],
        'token_top1_acc': eval_result['token_top1_acc'],
        'token_top5_acc': eval_result['token_top5_acc'],
        'char_similarity': eval_result['char_similarity'],
        'contains_reference': eval_result['contains_reference'],
        'contains_prediction': eval_result['contains_prediction'],
        'exact_match': eval_result['exact_match'],
        'llm_semantic_score': eval_result['llm_semantic_score'],
        'llm_used_fallback': llm_result['used_fallback'],
        'top1_gloss_sequence': ' '.join(top1_glosses),
        'target_gloss_sequence': ' '.join([gloss_to_text(g) for g in sentence_item['gloss_ids']])
    }
    all_sentence_rows.append(row)
    detailed_results.append({
        'sentence_item': sentence_item,
        'token_results': token_results,
        'llm_result': llm_result,
        'eval_result': eval_result
    })

summary_df = pd.DataFrame(all_sentence_rows)
display(summary_df)

aggregate = {
    'n_sentences': len(summary_df),
    'avg_token_top1_acc': float(summary_df['token_top1_acc'].mean()),
    'avg_token_top5_acc': float(summary_df['token_top5_acc'].mean()),
    'avg_char_similarity': float(summary_df['char_similarity'].mean()),
    'contains_reference_rate': float(summary_df['contains_reference'].mean()),
    'sentence_exact_match_rate': float(summary_df['exact_match'].mean()),
    'llm_fallback_rate': float(summary_df['llm_used_fallback'].mean())
}
if summary_df['llm_semantic_score'].notna().any():
    aggregate['avg_llm_semantic_score'] = float(summary_df['llm_semantic_score'].dropna().mean())

print('\n' + '=' * 90)
print('Aggregate metrics')
for k, v in aggregate.items():
    if isinstance(v, float):
        print(f'{k}: {v:.4f}')
    else:
        print(f'{k}: {v}')

print('\nSummary dataframe is available as: summary_df')
print('If you want to save it manually later, run: summary_df.to_csv(...)')

Sentence #1
Reference : 我喜歡蘋果
Gloss IDs : 1928 -> 2462 -> 0514


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Top1 gloss       : 2913:恐怕 / 2913:恐怕 / 0289:新
LLM output       : 老师喜欢苹果
Reference norm   : 我喜欢苹果
Prediction norm  : 老师喜欢苹果
Top1 acc         : 0.00%
Top5 acc         : 66.67%
Char sim         : 66.67%
Contains ref     : False
LLM status : API success


,position,target_gloss_id,target_gloss_zh,top1_id,top1_zh,top1_prob,top5_ids,top5_zh,top1_hit,top5_hit,sample_id
0,1,1928,我,2913,恐怕,0.3446,2913 | 2680 | 1597 | 2407 | 2835,恐怕 | 赌气 | 老师 | 矮 | 衣服,False,False,P10_1928_front
1,2,2462,喜欢,2913,恐怕,0.2316,2913 | 5807 | 2462 | 2680 | 2835,恐怕 | 渴 | 喜欢 | 赌气 | 衣服,False,True,P10_2462_front
2,3,0514,苹果,0289,新,0.4507,0289 | 0514 | 1270 | 5008 | 4878,新 | 苹果 | 写字 | 朋友 | 干净,False,True,P10_0514_front


Sentence #2
Reference : 母親買麵包
Gloss IDs : 2832 -> 3993 -> 2985


...

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
/usr/local/lib/python3.12/dist-packages/google/protobuf/sy

Top1 gloss       : 6056:没事 / 2678:树 / 0396:父亲 / 5478:和 / 1818:伯父 / 4870:做 / 4870:做 / 5753:椅子 / 4002:哥哥 / 2533:茶
LLM output       : 但是大雨父亲和伯父坐坐学校哥哥茶
Reference norm   : 因为大雨父亲和伯父坐在家喝茶
Prediction norm  : 但是大雨父亲和伯父坐坐学校哥哥茶
Top1 acc         : 40.00%
Top5 acc         : 100.00%
Char sim         : 56.25%
Contains ref     : False
LLM status : API success


,position,target_gloss_id,target_gloss_zh,top1_id,top1_zh,top1_prob,top5_ids,top5_zh,top1_hit,top5_hit,sample_id
0,1,5780,因为,6056,没事,0.0945,6056 | 6645 | 0930 | 1688 | 5780,没事 | 蓝 | 但是 | 坏 | 因为,False,True,P10_5780_front
1,2,0639,大雨,2678,树,0.4165,2678 | 0639 | 3422 | 5781 | 4944,树 | 大雨 | 篮球 | 高 | 眼镜,False,True,P10_0639_front
2,3,0396,父亲,0396,父亲,0.1874,0396 | 5807 | 2462 | 1879 | 5736,父亲 | 渴 | 喜欢 | 水 | 紫,True,True,P10_0396_front
3,4,5478,和,5478,和,0.8238,5478 | 6526 | 5753 | 0526 | 6488,和 | 参加 | 椅子 | 学生 | 教室,True,True,P10_5478_front
4,5,1818,伯父,1818,伯父,0.3530,1818 | 6343 | 5855 | 3185 | 5807,伯父 | 黄 | 红 | 睡觉 | 渴,True,True,P10_1818_front
5,6,2732,坐,4870,做,0.3040,4870 | 6363 | 2732 | 2039 | 5311,做 | 紧张 | 坐 | 在 | 对不起,False,True,P10_2732_front
6,7,2039,在,4870,做,0.4399,4870 | 2732 | 2039 | 6363 | 3993,做 | 坐 | 在 | 紧张 | 买,False,True,P10_2039_front
7,8,5726,家,5753,椅子,0.4227,5753 | 1251 | 5726 | 0526 | 5466,椅子 | 学校 | 家 | 学生 | 图书馆,False,True,P10_5726_front
8,9,3504,喝,4002,哥哥,0.3814,4002 | 3149 | 3504 | 3735 | 4979,哥哥 | 姐姐 | 喝 | 等 | 弟弟,False,True,P10_3504_front
9,10,2533,茶,2533,茶,0.7285,2533 | 1752 | 5726 | 3422 | 5753,茶 | 讲课 | 家 | 篮球 | 椅子,True,True,P10_2533_front


,sentence_id,reference_text,predicted_sentence,reference_norm,predicted_norm,token_top1_acc,token_top5_acc,char_similarity,contains_reference,contains_prediction,exact_match,llm_semantic_score,llm_used_fallback,top1_gloss_sequence,target_gloss_sequence
0,1,我喜歡蘋果,老师喜欢苹果,我喜欢苹果,老师喜欢苹果,0.000000,0.666667,0.666667,0,0,0,None,False,恐怕 恐怕 新,我 喜欢 苹果
1,2,母親買麵包,父亲买家,母亲买面包,父亲买家,0.333333,0.666667,0.400000,0,0,0,None,False,伯父 买 家,母亲 买 面包
2,3,老師幫助學生,老师帮助学生,老师帮助学生,老师帮助学生,0.333333,1.000000,1.000000,1,1,1,None,False,赌气 参加 学生,老师 帮助 学生
3,4,朋友找醫生,朋友公园说,朋友找医生,朋友公园说,0.666667,0.666667,0.400000,0,0,0,None,False,朋友 找 知道,朋友 找 医生
4,5,明天大家休息,以后我们休息,明天大家休息,以后我们休息,1.000000,1.000000,0.333333,0,0,0,None,False,明天 大家 休息,明天 大家 休息
5,6,昨天考試緊張,可以考试紧张,昨天考试紧张,可以考试紧张,0.666667,0.666667,0.666667,0,0,0,None,False,白 考试 紧张,昨天 考试 紧张
6,7,今天我去超市買牛奶和雞蛋,今天恐怕去超市买牛奶和鸡蛋,今天我去超市买牛奶和鸡蛋,今天恐怕去超市买牛奶和鸡蛋,0.375000,0.875000,0.846154,0,0,0,None,False,饿 恐怕 去 大雪 买 黑 和 水果,今天 我 去 超市 买 牛奶 和 鸡蛋
7,8,因為大雨，父親和伯父坐在家喝茶,但是大雨父亲和伯父坐坐学校哥哥茶,因为大雨父亲和伯父坐在家喝茶,但是大雨父亲和伯父坐坐学校哥哥茶,0.400000,1.000000,0.562500,0,0,0,None,False,没事 树 父亲 和 伯父 做 做 椅子 哥哥 茶,因为 大雨 父亲 和 伯父 坐 在 家 喝 茶



Aggregate metrics
n_sentences: 8
avg_token_top1_acc: 0.4719
avg_token_top5_acc: 0.8177
avg_char_similarity: 0.6094
contains_reference_rate: 0.1250
sentence_exact_match_rate: 0.1250
llm_fallback_rate: 0.0000

Summary dataframe is available as: summary_df
If you want to save it manually later, run: summary_df.to_csv(...)


In [ ]:
inspect_index = 0
item = detailed_results[inspect_index]
print('Reference sentence :', item['sentence_item']['reference_text'])
print('Predicted sentence :', item['llm_result']['sentence'])
print()
for token in item['token_results']:
    print(f"Position {token['position']} | target={token['target_gloss_id']} {token['target_gloss_zh']} | sample={token['sample_id']}")
    for cand in token['top5']:
        print(f"  Top{cand['rank']}: {cand['gloss_id']} {cand['gloss_zh']} p={cand['prob']:.4f}")
    print()

Reference sentence : 我喜歡蘋果
Predicted sentence : 老师喜欢苹果

Position 1 | target=1928 我 | sample=P10_1928_front
  Top1: 2913 恐怕 p=0.3446
  Top2: 2680 赌气 p=0.2876
  Top3: 1597 老师 p=0.0982
  Top4: 2407 矮 p=0.0549
  Top5: 2835 衣服 p=0.0270

Position 2 | target=2462 喜欢 | sample=P10_2462_front
  Top1: 2913 恐怕 p=0.2316
  Top2: 5807 渴 p=0.0948
  Top3: 2462 喜欢 p=0.0788
  Top4: 2680 赌气 p=0.0681
  Top5: 2835 衣服 p=0.0678

Position 3 | target=0514 苹果 | sample=P10_0514_front
  Top1: 0289 新 p=0.4507
  Top2: 0514 苹果 p=0.2333
  Top3: 1270 写字 p=0.0731
  Top4: 5008 朋友 p=0.0348
  Top5: 4878 干净 p=0.0333

